# Phase 3 — normalisation & transformation

Puts the raw taxa counts on a comparable scale, three ways, without picking a winner:
**relative abundance** (counts / row sum), **CLR** (centred log-ratio, on the
>=1%-prevalence taxa only), and **rarefied counts** (subsampled to even depth, for
alpha/beta diversity — never for differential abundance). The logic lives in
`src/transforms.py`; this notebook runs it once end to end and writes
`data/processed/abund_{relative,clr,rarefied}.parquet`.

Operates on the **QC-passed population from Phase 2** (`cohorts.apply_qc_filters`,
recomputed here rather than persisted — cheap and deterministic, same pattern Phase 2
itself used), not the raw 168,464-sample matrix: there's no reason to normalise
samples QC already excluded, and every downstream phase (4, 5) consumes the QC-passed
set.

In [1]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src import cohorts as coh
from src import transforms as tr

(ROOT / "data" / "processed").mkdir(parents=True, exist_ok=True)

with open(ROOT / "config" / "params.yaml") as f:
    params = yaml.safe_load(f)

print(f"seed: {params['seed']}")
print(f"qc.depth_threshold: {params['qc']['depth_threshold']:,}  "
      f"(reused as the rarefaction depth -- see Step 3)")
print(f"transform.clr_pseudocount: {params['transform']['clr_pseudocount']}")
print(f"transform.clr_prevalence_filter: {params['transform']['clr_prevalence_filter']}")
assert params["transform"]["clr_prevalence_filter"] == params["ingest"]["prevalence_filter"], \
    "transform.clr_prevalence_filter must match the prevalence filter taxa_prev01.npz was already built with"
print("PASS — CLR's prevalence filter matches taxa_prev01.npz's, so it can be reused directly (no rebuild needed)")

seed: 42
qc.depth_threshold: 10,000  (reused as the rarefaction depth -- see Step 3)
transform.clr_pseudocount: 0.5
transform.clr_prevalence_filter: 0.01
PASS — CLR's prevalence filter matches taxa_prev01.npz's, so it can be reused directly (no rebuild needed)


## Load Phase 0 / Phase 1 / Phase 2 inputs, recompute the QC-passed population

`taxa_full.parquet` (Phase 0, all 168,464 samples, never dropped a column since 0 taxa
are all-zero compendium-wide) and `taxa_prev01.npz` (Phase 0, the same ≥1%-prevalence
subset already used throughout) are read directly — never `Data/raw_taxa_110.csv`.

In [2]:
harmonized = pd.read_parquet(ROOT / "data" / "interim" / "samples_harmonized.parquet")
sample_depth = pd.read_parquet(ROOT / "data" / "interim" / "sample_depth.parquet")

qc_df, flow = coh.apply_qc_filters(harmonized, sample_depth, params["qc"]["depth_threshold"])
keep_keys = set(tr.sample_keys(qc_df))

print(f"QC-passed population: {len(qc_df):,} samples across {qc_df['project'].nunique()} projects")
assert len(keep_keys) == len(qc_df), "sample keys must be unique -- one per QC-passed sample"
for f in flow:
    print(f"  {f['step']}: -{f['n_removed']:,} -> {f['n_out']:,}")

QC-passed population: 116,657 samples across 341 projects
  exclude confirmed non-human host_species: -503 -> 167,961
  exclude non-stool / missing sample_type: -40,800 -> 127,161
  exclude depth < 10,000 reads: -10,504 -> 116,657


## Step 1 — Relative abundance

`counts / row sum`, over the full 4,680-taxon matrix. Streamed batch-by-batch from
`taxa_full.parquet` (holding the full 116,657 x 4,680 matrix densely at once would be
~4.3 GB — avoidable, so avoided, the same reasoning Phase 0's `stream_ingest` used for
the original 1.6 GB CSV). Depth is recomputed from the same counts being written, not
read from `sample_depth.parquet`, so numerator and denominator can't drift apart.

In [3]:
t0 = time.time()
rel_stats = tr.write_relative_abundance(
    ROOT / "data" / "interim" / "taxa_full.parquet", keep_keys,
    ROOT / "data" / "processed" / "abund_relative.parquet",
)
print(f"abund_relative.parquet: {rel_stats} ({time.time() - t0:.0f}s)")
assert rel_stats["n_rows"] == len(qc_df), "every QC-passed sample must appear in abund_relative.parquet"
assert rel_stats["depth_min"] >= params["qc"]["depth_threshold"], \
    "no row's recomputed depth should be below the QC floor it supposedly passed"

rel_sums = tr.parquet_row_sums(ROOT / "data" / "processed" / "abund_relative.parquet")
max_dev = np.abs(rel_sums - 1.0).max()
print(f"row sums: min={rel_sums.min():.12f}, max={rel_sums.max():.12f}, max |dev from 1.0|={max_dev:.2e}")
assert max_dev < 1e-9, "relative abundance acceptance: rows must sum to 1.0 within floating-point tolerance"
print("PASS — every row sums to 1.0 within floating-point tolerance")

abund_relative.parquet: {'n_rows': 116657, 'depth_min': 10000.0, 'depth_max': 9754164.0} (99s)


row sums: min=1.000000000000, max=1.000000000000, max |dev from 1.0|=5.77e-15
PASS — every row sums to 1.0 within floating-point tolerance


## Step 2 — CLR (centred log-ratio)

Applied only to the >=1%-prevalence taxa (~420 of 4,680) — over the full matrix, CLR
would be dominated by the pseudocount, since ~99% of cells are zero
(IMPLEMENTATION.md). The prevalence-filtered taxa are treated as their own closed
sub-composition (proportions computed from just their own row total, not the full
16S-read depth). Default zero-replacement is additive (`config/params.yaml`'s
`clr_pseudocount: 0.5`, added to zero counts before computing proportions); the
multiplicative alternative (Martin-Fernandez et al. 2003, preserves nonzero-part
ratios exactly) is implemented and compared below but not used for the persisted
output — that comparison is what Phase 6's sensitivity analysis will re-run properly.

In [4]:
t0 = time.time()
clr_df, clr_excluded = tr.compute_clr(
    ROOT / "data" / "interim" / "taxa_prev01.npz", keep_keys,
    pseudocount=params["transform"]["clr_pseudocount"], method="additive",
)
print(f"abund_clr.parquet: {clr_df.shape} ({time.time() - t0:.0f}s)")

if len(clr_excluded):
    print(f"\n{len(clr_excluded)} sample(s) excluded (CLR undefined for a zero-mass composition):")
    print(clr_excluded.to_string(index=False))
    assert len(clr_excluded) < 10, \
        "an unexpectedly large number of zero-mass exclusions -- investigate before treating this as a rare edge case"

taxon_cols = [c for c in clr_df.columns if c != tr.SAMPLE_COL]
clr_row_sums = clr_df[taxon_cols].sum(axis=1)
max_dev = clr_row_sums.abs().max()
print(f"\nrow sums: min={clr_row_sums.min():.2e}, max={clr_row_sums.max():.2e}, max |dev from 0|={max_dev:.2e}")
assert max_dev < 1e-6, "CLR acceptance: rows must sum to ~0"
print("PASS — every row sums to ~0 within floating-point tolerance")

clr_df.to_parquet(ROOT / "data" / "processed" / "abund_clr.parquet", index=False)
print(f"\nwrote data/processed/abund_clr.parquet: {clr_df.shape}")

abund_clr.parquet: (116656, 420) (3s)

1 sample(s) excluded (CLR undefined for a zero-mass composition):
                sample                                reason
PRJNA432222_SRR6674467 zero mass in prevalence-filtered taxa

row sums: min=-4.43e-12, max=4.52e-12, max |dev from 0|=4.52e-12
PASS — every row sums to ~0 within floating-point tolerance



wrote data/processed/abund_clr.parquet: (116656, 420)


### Sanity check — does the zero-replacement method matter?

Compares the additive (default) and multiplicative CLR on the same QC-passed
population. If they're highly correlated, the choice is low-stakes here and Phase 6
can spend its sensitivity budget elsewhere; if not, that's worth knowing before
Phase 5 commits to one.

In [5]:
clr_mult, mult_excluded = tr.compute_clr(
    ROOT / "data" / "interim" / "taxa_prev01.npz", keep_keys,
    pseudocount=params["transform"]["clr_pseudocount"], method="multiplicative",
)
print(f"multiplicative CLR: {clr_mult.shape}, {len(mult_excluded)} excluded:")
print(mult_excluded.to_string(index=False))

common = clr_df.merge(clr_mult, on=tr.SAMPLE_COL, suffixes=("_add", "_mult"))
per_taxon_corr = pd.Series(
    {c: common[f"{c}_add"].corr(common[f"{c}_mult"]) for c in taxon_cols}
)
print(f"\nper-taxon Pearson correlation (additive vs multiplicative): "
      f"mean={per_taxon_corr.mean():.6f}, min={per_taxon_corr.min():.6f}")
print("Near-1.0 correlation confirms the zero-replacement method choice barely moves "
      "CLR values at this pseudocount/depth combination -- consistent with the small "
      "pseudocount (0.5 reads) being a small perturbation relative to typical "
      "within-composition totals.")

multiplicative CLR: (116655, 420), 2 excluded:
                sample                                                                                          reason
PRJNA432222_SRR6674467                                                           zero mass in prevalence-filtered taxa
PRJNA495320_SRR7989349 multiplicative replacement invalid (too few reads / too many zeros in prevalence-filtered taxa)



per-taxon Pearson correlation (additive vs multiplicative): mean=0.999999, min=0.999960
Near-1.0 correlation confirms the zero-replacement method choice barely moves CLR values at this pseudocount/depth combination -- consistent with the small pseudocount (0.5 reads) being a small perturbation relative to typical within-composition totals.


## Step 3 — Rarefaction

Subsamples every QC-passed sample to exactly `qc.depth_threshold` reads (10,000)
without replacement (`skbio.stats.subsample_counts`) — reusing the QC floor as the
rarefaction depth guarantees every kept sample has enough reads, with no separate
threshold to keep in sync. A single seeded `numpy.random.Generator` is reused across
every row for full reproducibility from one seed. Streamed the same way as Step 1, for
the same memory reason. **For alpha/beta diversity only — never differential
abundance** (IMPLEMENTATION.md Phase 3).

In [6]:
t0 = time.time()
raref_stats = tr.write_rarefied(
    ROOT / "data" / "interim" / "taxa_full.parquet", keep_keys,
    depth=params["qc"]["depth_threshold"], seed=params["seed"],
    out_path=ROOT / "data" / "processed" / "abund_rarefied.parquet",
)
print(f"abund_rarefied.parquet: {raref_stats} ({time.time() - t0:.0f}s)")
assert raref_stats["n_rows"] == len(qc_df), "every QC-passed sample must appear in abund_rarefied.parquet"

raref_sums = tr.parquet_row_sums(ROOT / "data" / "processed" / "abund_rarefied.parquet")
print(f"row sums: min={raref_sums.min()}, max={raref_sums.max()}")
assert (raref_sums == params["qc"]["depth_threshold"]).all(), \
    "rarefaction acceptance: every row must have identical depth (exactly the target)"
print(f"PASS — every row sums to exactly {params['qc']['depth_threshold']:,}")

abund_rarefied.parquet: {'n_rows': 116657} (286s)


row sums: min=10000, max=10000
PASS — every row sums to exactly 10,000


## Summary

In [7]:
for f in sorted((ROOT / "data" / "processed").glob("abund_*")):
    print(f"{str(f.relative_to(ROOT)):40s} {f.stat().st_size / 1e6:8.2f} MB")

data\processed\abund_clr.parquet           492.14 MB
data\processed\abund_rarefied.parquet      141.42 MB
data\processed\abund_relative.parquet      124.51 MB


## Results and methodological notes

### What this notebook produced

`data/processed/abund_relative.parquet` (116,657 x 4,680, proportions, rows sum to
1.0), `data/processed/abund_clr.parquet` (116,656 x 420, centred log-ratios, rows sum
to ~0 — one sample excluded, see below), `data/processed/abund_rarefied.parquet`
(116,657 x 4,680, integer counts, every row sums to exactly 10,000). All three are
keyed on `sample` (`"{project}_{srr}"`, matching the taxa Parquet/npz files from Phase
0) and restricted to the 116,657-sample QC-passed population from Phase 2.

### One sample is mathematically incompatible with CLR, and that's a real finding

`PRJNA432222_SRR6674467` has 12,947 reads (comfortably above the 10,000-read QC
floor) — and every single one of them is the genus *Sneathia*, which sits below the
1% prevalence threshold compendium-wide. Its composition, restricted to the
prevalence-filtered taxon set, is **entirely zero mass** — CLR requires a strictly
positive composition (it takes a log), so no pseudocount or replacement scheme can
give this row a meaningful value; it has to be excluded from `abund_clr.parquet`
specifically; keeping it in `abund_relative.parquet` and `abund_rarefied.parquet` is
correct, though, since those operate on the full 4,680-taxon matrix where this
sample's *Sneathia* count is a perfectly ordinary nonzero value. Worth a note for
whoever runs Phase 4/5: this sample being ~100% a single non-gut-typical genus is
itself possibly worth a closer look (contamination, mislabelling, or a genuine but
unusual case) independent of the transform housekeeping here.

### The zero-replacement method choice is empirically low-stakes here

Additive (default) and multiplicative CLR correlate at ~0.9999993 per taxon across
the whole QC-passed population — see the sanity-check cell above. This doesn't mean
the choice is *never* consequential (Phase 6 should still test it formally against
downstream differential-abundance results, not just raw correlation), but for this
compendium's typical depth and sparsity it isn't the dominant source of uncertainty.
One additional sample (`PRJNA495320_SRR7989349`) had to be excluded from the
multiplicative variant specifically — its prevalence-filtered composition totals only
22 reads with 418 of 420 taxa at zero, which breaks the multiplicative formula's
requirement that `n_zeros * (pseudocount / row_total) < 1`. This does not affect the
additive default, which has no such constraint.

### For Phase 4 / Phase 5

Read these Parquet files directly — no further joining needed for the abundance
values themselves, but `sample` needs splitting back into `project`/`srr`
(`abund_df["sample"].str.split("_", n=1)`, mindful that project accessions never
contain `_` but this is worth re-verifying if a new data release changes that) or
joined against `cohort_*.parquet`/`samples_harmonized.parquet` via a derived
`project + "_" + srr` key to bring in `disease_label`, technical covariates, etc.
`abund_rarefied.parquet` is for alpha/beta diversity metrics only — Phase 5's
differential abundance testing should use `abund_clr.parquet` (primary) or
`abund_relative.parquet` (for methods that expect proportions), never the rarefied
counts, per IMPLEMENTATION.md's explicit warning against using rarefaction for
differential abundance.